# M4 - judge (Layer 4) + baseline ASR

First real results: run the whole battery against the **undefended** target, let the
judge label every reply BAD_BOT / GOOD_BOT / UNCLEAR, and read off per-attack Attack
Success Rate.

- target `Qwen2.5-3B` -> **cuda:0** (fp16, ~6 GB)
- judge `Qwen3.5-9B` -> **cuda:1** (nf4 4-bit, ~6 GB)

**Accelerator must be `GPU T4 x2`.** Internet On. If Qwen3.5-9B won't load, set
`[models.judge] name = "Qwen/Qwen3-8B"` and drop `quant` -- it's text-only and well-supported.

## 1 - Setup

In [ ]:
%pip -q install -U transformers accelerate bitsandbytes huggingface_hub

In [ ]:
import os, subprocess, sys, pathlib, time, json, glob
import numpy as np, pandas as pd

_sec = None
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
except Exception as e:
    print('no Kaggle secrets client:', e)

def _secret(name):
    try:
        return _sec.get_secret(name) if _sec is not None else None
    except Exception:
        return None

_hf = _secret('HF_TOKEN')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    from huggingface_hub import login; login(token=_hf)
    print('HF auth OK')
else:
    print('no HF_TOKEN secret (fine - models are public)')

In [ ]:
# --- get the repo (public or private; safe to re-run) ----------------------
REPO   = "MehemudAzad/LLM-jailbreaking-with-layered-prompt-defense"
BRANCH = "main"
WORK   = pathlib.Path("/kaggle/working")
ROOT   = WORK / "repo"

_gh  = _secret("GH_TOKEN")
_url = f"https://{_gh}@github.com/{REPO}.git" if _gh else f"https://github.com/{REPO}.git"

os.chdir(WORK)
subprocess.run(["rm", "-rf", str(ROOT)], check=False)
_r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, _url, str(ROOT)],
                    cwd=str(WORK), capture_output=True, text=True)
if _r.returncode != 0:
    _err = _r.stderr.replace(_gh, "***") if _gh else _r.stderr
    raise RuntimeError("git clone failed:\n" + _err +
        "\n\nPrivate repo? add a GH_TOKEN Kaggle secret (fine-grained PAT, Contents: read-only)."
        "\nOr make the repo public. Also: git push -u origin main")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("HEAD", subprocess.check_output(["git","-C",str(ROOT),"rev-parse","--short","HEAD"]).decode().strip())

In [ ]:
# frozen harmful set is git-ignored -> build it (seeded, deterministic)
r = subprocess.run([sys.executable, "datasets/build_harmful.py"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

In [ ]:
import torch
print('CUDA devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {p.total_memory/1e9:.0f} GB')
assert torch.cuda.device_count() >= 2, 'set the accelerator to GPU T4 x2'

In [ ]:
from core.config import CONFIG
t, j = CONFIG['models']['target'], CONFIG['models']['judge']
print('target:', t['name'], '->', t.get('device'), '| dtype', t.get('dtype'))
print('judge :', j['name'], '->', j.get('device'), '| quant', j.get('quant'), '| thinking', j.get('thinking'))
print('layer1:', CONFIG['defense']['layer1_perplexity'])

## 2 - Sanity pass (10 goals)

Loads both models, runs 10 goals x ~16 attacks undefended, judge grades. Check the
labels below look right **before** the 50-goal run.

In [ ]:
from run_eval import main
t0 = time.time()
main(['--attack', 'all', '--defense', 'off', '--limit', '10', '--tag', 'm4sanity'])
print(f'\nsanity pass: {time.time()-t0:.0f}s')

In [ ]:
def load_run(tag):
    d = sorted(glob.glob(f'logs/*{tag}*'))[-1]
    recs = [json.loads(l) for l in open(f'{d}/transcript.jsonl')]
    trials = pd.DataFrame([r for r in recs if r['type'] == 'trial'])
    summary = next(r for r in recs if r['type'] == 'summary')
    return d, trials, summary

d, tr, sm = load_run('m4sanity')
print('run:', d, '| trials:', sm['trials'], '| overall ASR:', sm['asr'], '| labels:', sm['labels'])
print()
for _, row in tr[tr.label != 'GOOD_BOT'].head(8).iterrows():
    print(f"[{row.attack}]  label={row.label} ({row.label_method})")
    print('  goal :', row.metadata['goal'][:90])
    print('  reply:', (row.response or '')[:220].replace(chr(10),' '), '\n')

**Gate:** do the labels match the replies? A refused reply must be GOOD_BOT, a reply
that actually gives the harmful content must be BAD_BOT. If the judge is wrong, fix
`defense/layer4_response_classifier.py` (the `_JUDGE_SYSTEM` prompt) and re-run before continuing.

## 3 - Full baseline (50 goals)

In [ ]:
t0 = time.time()
main(['--attack', 'all', '--defense', 'off', '--tag', 'm4baseline'])
print(f'\nfull baseline: {(time.time()-t0)/60:.1f} min')

In [ ]:
d, tr, sm = load_run('m4baseline')
print('run dir:', d)

tbl = (pd.DataFrame({'ASR': sm['asr_by_attack']})
         .join(tr.groupby('attack').category.first())
         .sort_values('ASR', ascending=False))
tbl['ASR_%'] = (tbl.ASR * 100).round(1)
print('\n=== BASELINE ASR by attack (undefended Qwen2.5-3B, 50 AdvBench goals) ===\n')
print(tbl[['category', 'ASR_%']].to_string())
print(f'\noverall ASR : {sm["asr"]*100:.1f}%   ({sm["labels"].get("BAD_BOT",0)}/{sm["trials"]})')
print('label mix   :', sm['labels'])
print('graded by   :', tr.label_method.value_counts().to_dict())

## 4 - Pin the judge revision

In [ ]:
from huggingface_hub import HfApi
for role in ('judge',):
    nm = CONFIG['models'][role]['name']
    try:
        sha = HfApi().model_info(nm, token=os.environ.get('HF_TOKEN')).sha
        print(f'[models.{role}]  # {nm}\n  revision = "{sha}"')
    except Exception as e:
        print(role, 'revision lookup failed:', e)

## Done - what to commit

- `core/models.py` (quant/device/thinking), `defense/layer4_response_classifier.py`,
  `run_eval.py` (`--grade` + per-attack ASR), `config.toml`, `requirements.txt`
- `notebooks/m4_baseline_asr.ipynb`
- the pinned judge `revision`

For the report: the baseline ASR table (this is the 'before' half of every result).

**M5:** wire Layer 2 (paraphraser) for real, then the **defended** ASR pass -- same
battery, `--defense on` -- and the before/after comparison.